In [ ]:
# --- ensure repo-root cwd so data/ paths resolve from anywhere ---
import os, sys
from pathlib import Path
_r = Path.cwd()
while not (_r / 'requirements.txt').exists() and _r != _r.parent:
    _r = _r.parent
os.chdir(_r); sys.path.insert(0, str(_r / 'src'))

# Extract telemetry for the top 10 VINs (S3 Select)

The S3 data is Parquet **partitioned by date** (`year=/month=/day=/part-*.snappy.parquet`),
and the `vin` lives *inside* each file — so a given VIN can appear in any file.

Rather than download the whole dataset and filter locally, this notebook pushes a
`WHERE vin IN (...)` filter into S3 with **S3 Select** (server-side), runs it across
files in parallel, and prunes partitions outside the VINs' active date range. Only the
matching rows ever cross the network; they land in `data/extracted/` as Parquet.

**Before running:** fill in `.env`, then select the **Python (EULER_RUL)** kernel (top-right).

## 1. Setup — load credentials and connect to S3

In [ ]:
import os
import re
import json
import threading
from pathlib import Path
from collections import Counter
from concurrent.futures import ThreadPoolExecutor, as_completed

import boto3
import pandas as pd
from botocore.config import Config
from dotenv import load_dotenv
from tqdm.auto import tqdm

load_dotenv()  # reads .env

BUCKET = os.environ["S3_BUCKET"]
PREFIX = os.environ.get("S3_PREFIX", "")
DATA_DIR = Path(os.environ.get("DATA_DIR", "data"))
OUT_DIR = DATA_DIR / "extracted"
OUT_DIR.mkdir(parents=True, exist_ok=True)

MAX_WORKERS = 24  # parallel S3 Select requests

# Bump the connection pool so all workers get a socket; boto3 clients are thread-safe.
s3 = boto3.client("s3", config=Config(max_pool_connections=MAX_WORKERS + 4))

s3.head_bucket(Bucket=BUCKET)
print(f"Connected to s3://{BUCKET}/{PREFIX}")

## 2. Select the oldest 10 VINs and their active date range

"Oldest" means the VINs first seen earliest in the telemetry (smallest `first_seen_date`).
The union of their `first_seen`/`last_seen` dates bounds which date partitions we scan.

In [ ]:
TOP_N = 10

manifest = pd.read_csv("data/manifests/euler_vin_manifest.csv")
# "Oldest" = vehicles seen earliest in the telemetry (smallest first_seen_date).
manifest["first_seen_date"] = pd.to_datetime(manifest["first_seen_date"])
manifest["last_seen_date"] = pd.to_datetime(manifest["last_seen_date"])
top = manifest.sort_values("first_seen_date").head(TOP_N)
top_vins = top["vin"].tolist()

DATE_START = top["first_seen_date"].min().date()
DATE_END = top["last_seen_date"].max().date()

print(f"Selected {len(top_vins)} oldest VINs; active range {DATE_START} → {DATE_END}")
top[["vin", "first_seen_date", "last_seen_date", "total_records"]]

## 3. List the source objects (date-pruned)

Lists every Parquet object under the prefix, keeping only those whose `year=/month=/day=`
partition falls inside the VINs' active range.

In [ ]:
_part_re = re.compile(r"year=(\d{4})/month=(\d{2})/day=(\d{2})")


def partition_date(key: str):
    """Return the partition date encoded in the key, or None if absent."""
    m = _part_re.search(key)
    if not m:
        return None
    y, mo, d = map(int, m.groups())
    return pd.Timestamp(y, mo, d).date()


paginator = s3.get_paginator("list_objects_v2")
source_keys = []
for page in paginator.paginate(Bucket=BUCKET, Prefix=PREFIX):
    for obj in page.get("Contents", []):
        key = obj["Key"]
        if not key.endswith(".parquet"):
            continue
        d = partition_date(key)
        if d is None or DATE_START <= d <= DATE_END:
            source_keys.append(key)

print(f"{len(source_keys):,} parquet objects to scan within the date range")

## 4. Extract matching rows with S3 Select (parallel, server-side)

Each worker sends `SELECT * WHERE vin IN (...)` to one object; S3 returns only the
matching rows as JSON. Matches for that object are written to `data/extracted/` as a
single Parquet file (named after the source key). Re-running skips objects already done.

In [ ]:
# Full column set (S3 Select omits null fields, so we reindex to keep schemas consistent).
ALL_COLS = [
    "eventAt", "requestUUID", "vin", "registrationNumber", "imei", "chassisNumber",
    "vehicleLastUpdatedAt", "locationLastUpdatedAt", "batterySoc", "batterySoh",
    "odometer", "location", "vehicleMode", "batteryTemperature", "batteryVoltage",
]

_vin_list_sql = ", ".join(f"'{v}'" for v in top_vins)
EXPRESSION = f"SELECT * FROM s3object s WHERE s.vin IN ({_vin_list_sql})"


def out_path_for(key: str) -> Path:
    """Flatten a source key into a single output filename."""
    rel = key[len(PREFIX):] if key.startswith(PREFIX) else key
    return OUT_DIR / (rel.replace("/", "__").replace(".parquet", "") + ".parquet")


def extract_one(key: str):
    """S3 Select one object, write matches to Parquet. Returns (n_rows, per-vin Counter)."""
    out_path = out_path_for(key)
    if out_path.exists():
        return None  # already done

    resp = s3.select_object_content(
        Bucket=BUCKET, Key=key, ExpressionType="SQL", Expression=EXPRESSION,
        InputSerialization={"Parquet": {}},
        OutputSerialization={"JSON": {"RecordDelimiter": "\n"}},
    )

    buf = bytearray()
    for event in resp["Payload"]:
        if "Records" in event:
            buf += event["Records"]["Payload"]

    rows = [json.loads(line) for line in buf.decode().splitlines() if line.strip()]
    if not rows:
        return (0, Counter())

    df = pd.DataFrame(rows).reindex(columns=ALL_COLS)
    df.to_parquet(out_path, index=False)
    return (len(df), Counter(df["vin"]))


totals = Counter()
n_rows = n_files_with_data = errors = 0
lock = threading.Lock()

with ThreadPoolExecutor(max_workers=MAX_WORKERS) as pool:
    futures = {pool.submit(extract_one, k): k for k in source_keys}
    for fut in tqdm(as_completed(futures), total=len(futures), desc="S3 Select", unit="obj"):
        try:
            result = fut.result()
        except Exception as e:  # noqa: BLE001
            errors += 1
            if errors <= 5:
                print(f"  error on {futures[fut]}: {type(e).__name__}: {e}")
            continue
        if result is None:
            continue
        rows, per_vin = result
        with lock:
            n_rows += rows
            if rows:
                n_files_with_data += 1
            totals.update(per_vin)

print(f"\nDone. {n_rows:,} rows across {n_files_with_data:,} files. Errors: {errors}")
print("Rows per VIN:")
for vin in top_vins:
    print(f"  {vin}: {totals.get(vin, 0):,}")

## 5. Load the extracted data

Read everything in `data/extracted/` back as one DataFrame. For very large pulls,
prefer a partitioned read with `pyarrow.dataset` or process VIN-by-VIN instead.

In [ ]:
import pyarrow.dataset as ds

dataset = ds.dataset(OUT_DIR, format="parquet")
df = dataset.to_table().to_pandas()
df["eventAt"] = pd.to_datetime(df["eventAt"], unit="ms")  # epoch ms -> datetime
df = df.sort_values(["vin", "eventAt"]).reset_index(drop=True)

print(df.shape)
df.head()